# Length of Hospital Stay Prediction Pipeline
## Jett Guitteau
### This project is a self-guided part of a learning experience at Qualees to explore machine learning methods and compare a local python machine learning pipeline with their TweenMe platform which can augment data to enhance model performance. This project focuses on regression. 

This project utilizes a dataset that includes around 100,000 records of hospital stays for patients with diabetes. 

The features of the dataset include patient race, gender, age, their admission type, discharge disposition, key diabetes lab values, diagnoses codes, and medication information. The goal of this model is to predict length of stay at admission, so columns that reflect information gathered during the stay or at discharge are removed to avoid data leakage. 

The dataset can be found on the UC Irvine Machine Learning Repository: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [2]:
diabetes = pd.read_csv("diabetic_data.csv")
diabetes.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


First, the columns reflecting information gathered during or after the visit are to be dropped.

In [3]:
meds = diabetes.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
print(len(meds), meds)

during_or_after = [
    "num_lab_procedures", "num_procedures", "num_medications", "number_diagnoses",
    "max_glu_serum", "A1Cresult", "discharge_disposition_id", "readmitted",
    "change", "diabetesMed", "diag_2", "diag_3"
] + meds

diabetes = diabetes.drop(columns=during_or_after)

23 ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


Since diag_1 is the coded primary diagnosis, it will be used as a proxy for the admitting diagnosis, so it will be kept while the secondary diagnoses are dropped. The ICD codes will be simplified to their main categories, removing the smaller details given by the numbers after the decimal 
(ex. 300.01 and 300.03 will merge to 300).

In [4]:
diabetes["diag_1"] = diabetes["diag_1"].str[:3]
diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   encounter_id         101766 non-null  int64
 1   patient_nbr          101766 non-null  int64
 2   race                 101766 non-null  str  
 3   gender               101766 non-null  str  
 4   age                  101766 non-null  str  
 5   weight               101766 non-null  str  
 6   admission_type_id    101766 non-null  int64
 7   admission_source_id  101766 non-null  int64
 8   time_in_hospital     101766 non-null  int64
 9   payer_code           101766 non-null  str  
 10  medical_specialty    101766 non-null  str  
 11  number_outpatient    101766 non-null  int64
 12  number_emergency     101766 non-null  int64
 13  number_inpatient     101766 non-null  int64
 14  diag_1               101766 non-null  str  
dtypes: int64(8), str(7)
memory usage: 11.6 MB


In [5]:
'''Convert nominal admission columns to categorical data type. No need to map the exact names from the IDS as 
we are only interested in treating them as categorical variables. They will be one-hot encoded later 
in the pipeline. However, the various values that reflect a null value will be changed to NaN.'''

nominal_cols = ['admission_type_id', 'admission_source_id']
diabetes[nominal_cols] = diabetes[nominal_cols].astype('category') 
diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   encounter_id         101766 non-null  int64   
 1   patient_nbr          101766 non-null  int64   
 2   race                 101766 non-null  str     
 3   gender               101766 non-null  str     
 4   age                  101766 non-null  str     
 5   weight               101766 non-null  str     
 6   admission_type_id    101766 non-null  category
 7   admission_source_id  101766 non-null  category
 8   time_in_hospital     101766 non-null  int64   
 9   payer_code           101766 non-null  str     
 10  medical_specialty    101766 non-null  str     
 11  number_outpatient    101766 non-null  int64   
 12  number_emergency     101766 non-null  int64   
 13  number_inpatient     101766 non-null  int64   
 14  diag_1               101766 non-null  str     
dtypes: category

In [6]:
'''Treat various values that reflect a null value as NaN. For example, admission_type_id has 8 codes, 
with 5 meaning Not Available, 6 meaning NULL, and 8 meaning Not Mapped. All of these mean missing or unavailable values and 
should be treated as NaN. Similarly, discharge_disposition_id and admission_source_id have codes that reflect missing or 
unavailable values, which should also be treated as NaN.'''

diabetes["admission_type_id"] = diabetes["admission_type_id"].replace({5: np.nan, 6: np.nan, 8: np.nan})
diabetes["admission_source_id"] = diabetes["admission_source_id"].replace({9: np.nan, 15: np.nan, 17: np.nan, 20: np.nan, 21: np.nan})

In [7]:
for column in diabetes.columns:
    diabetes[column] = diabetes[column].replace("?", np.nan)
for column in diabetes.columns:
    if (diabetes[column].isna().sum() / diabetes.shape[0] * 100).round(2) > 0:
        print(f'Column "{column}" has missing values. Percentage of missing values:')
        print((diabetes[column].isna().sum() / diabetes.shape[0] * 100).round(2))

Column "race" has missing values. Percentage of missing values:
2.23
Column "weight" has missing values. Percentage of missing values:
96.86
Column "admission_type_id" has missing values. Percentage of missing values:
10.22
Column "admission_source_id" has missing values. Percentage of missing values:
6.94
Column "payer_code" has missing values. Percentage of missing values:
39.56
Column "medical_specialty" has missing values. Percentage of missing values:
49.08
Column "diag_1" has missing values. Percentage of missing values:
0.02


In [8]:
diabetes = diabetes.drop(columns="weight")
diabetes.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,number_outpatient,number_emergency,number_inpatient,diag_1
0,2278392,8222157,Caucasian,Female,[0-10),NaN,1,1,NaN,Pediatrics-Endocrinology,0,0,0,250
1,149190,55629189,Caucasian,Female,[10-20),1,7,3,NaN,NaN,0,0,0,276
2,64410,86047875,AfricanAmerican,Female,[20-30),1,7,2,NaN,NaN,2,0,1,648
3,500364,82442376,Caucasian,Male,[30-40),1,7,2,NaN,NaN,0,0,0,8
4,16680,42519267,Caucasian,Male,[40-50),1,7,1,NaN,NaN,0,0,0,197


In [9]:
diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   encounter_id         101766 non-null  int64   
 1   patient_nbr          101766 non-null  int64   
 2   race                 99493 non-null   str     
 3   gender               101766 non-null  str     
 4   age                  101766 non-null  str     
 5   admission_type_id    91370 non-null   category
 6   admission_source_id  94699 non-null   category
 7   time_in_hospital     101766 non-null  int64   
 8   payer_code           61510 non-null   str     
 9   medical_specialty    51817 non-null   str     
 10  number_outpatient    101766 non-null  int64   
 11  number_emergency     101766 non-null  int64   
 12  number_inpatient     101766 non-null  int64   
 13  diag_1               101745 non-null  str     
dtypes: category(2), int64(6), str(6)
memory usage: 9.5 MB


Next we will check for and remove all duplicate encounters, as each should be unique. 

In [10]:
diabetes["encounter_id"].duplicated().sum()

np.int64(0)

The encounter_id is already unique, therefore the column can be dropped as it won't be used for the model. Next we will check for duplicate patients, which would be fine to have since the encounter_id uniqueness indicates that these would be separate hospital stays for the patients seen more than once. 

In [11]:
diabetes["patient_nbr"].duplicated().sum()

np.int64(30248)

A group-aware split could be considered to maintain these rows, but for simplicity the duplicates will be removed to keep only one record per patient. After this drop, there will still be over 70,000 records.

In [12]:
diabetes = diabetes.drop_duplicates(subset="patient_nbr")
diabetes.shape

(71518, 14)

In [ ]:
# Dropping the now unnecessary columns: "encounter_id" and "patient_nbr"
diabetes = diabetes.drop(columns=["encounter_id", "patient_nbr"])
diabetes.shape

(71518, 12)

In [ ]:
# Going back to the two columns with high missing value counts: "medical_specialty" and "payer_code"
diabetes["medical_specialty"].value_counts()

medical_specialty
InternalMedicine          10919
Family/GeneralPractice     5118
Emergency/Trauma           4465
Cardiology                 4266
Surgery-General            2221
                          ...  
SportsMedicine                1
Speech                        1
Perinatology                  1
Neurophysiology               1
Resident                      1
Name: count, Length: 70, dtype: int64

In [14]:
X = diabetes.drop(columns="time_in_hospital")
y = diabetes["time_in_hospital"]
print(X.columns.tolist())

['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1']
